# COLET pipeline: LR vs XGBoost (Colab)

Run the cells **in order**. Each stage shows its output; check it before running the next.
Before the first run, upload the folder `parquet/` (from `data/colet/parquet/`, 847 MB) to
Google Drive, e.g. `MyDrive/csrp/colet/parquet/`.

In [ ]:
import getpass, subprocess, os
REPO = "elmigue21/csrp"
if not os.path.exists("/content/csrp"):
    token = getpass.getpass("GitHub token (press Enter if the repo is public): ").strip()
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", url, "/content/csrp"], capture_output=True, text=True)
    if r.returncode == 0:
        # never leave the token in .git/config
        subprocess.run(["git", "-C", "/content/csrp", "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)
        print("cloned")
    else:
        print(r.stderr.replace(token, "***") if token else r.stderr)
else:
    # a private repo needs the token re-entered (delete /content/csrp and rerun this cell);
    # the token is not stored in .git/config
    r = subprocess.run(["git", "-C", "/content/csrp", "pull"], capture_output=True, text=True)
    print(r.stdout, r.stderr)

In [ ]:
import importlib.metadata as md, os, subprocess, sys
pins = dict(l.strip().split("==") for l in open("/content/csrp/requirements-colab.txt") if "==" in l)

def installed(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

wrong = {k: v for k, v in pins.items() if installed(k) != v}
if wrong:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/csrp/requirements-colab.txt"], check=True)
    print("Installed pinned versions; restarting the runtime. Then run again from the Drive-mount cell.")
    os.kill(os.getpid(), 9)
print("All package versions match requirements-colab.txt")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
os.environ["COLET_DATA_DIR"] = "/content/drive/MyDrive/csrp/colet/parquet"   # edit if needed
os.environ["COLET_OUT_DIR"] = "/content/drive/MyDrive/csrp/outputs/colet"
sys.path.insert(0, "/content/csrp/src")
print(len(os.listdir(os.environ["COLET_DATA_DIR"])), "files found (expect 566)")

## Stage 1: features, exclusion, retention (P0 to P14)

In [ ]:
from config import DATA_DIR, CACHE_DIR
from run_colet import prepare
prep = prepare(DATA_DIR, CACHE_DIR)
print("analysis participants:", len(prep["participants"]), "(expect 45)")
prep["retention"]

## Stage 2: label-blind checks (sanity per activity; NASA-RTLX)

In [ ]:
from checks import manipulation_check
from data import load_annotation
display(prep["sanity"])
print("Features used in P1:", prep["features_used"])
manipulation_check(load_annotation(DATA_DIR), prep["participants"])

## Stage 3: run the analysis once, save every table, then show the results
`main` runs P1 and P2 (LR vs XGBoost, LOPO) **once** and writes each analysis's tables to
`OUT_DIR/tables` as soon as it finishes, so a disconnect during P2 keeps P1. It takes a while
(features are cached from Stage 1). Everything below is read back from the saved files.

In [ ]:
from config import OUT_DIR
from run_colet import main
meta = main(DATA_DIR, OUT_DIR)
{k: v for k, v in meta.items() if k != "analyses"}

In [ ]:
import pandas as pd
tables = OUT_DIR / "tables"
for name in ("05_results", "06_comparison", "07_importance_P1_shap_lr", "07_importance_P1_shap_xgb",
             "07_importance_P1_coef_lr", "07_importance_P1_perm_lr", "07_importance_P1_perm_xgb"):
    print(name)
    display(pd.read_csv(tables / f"{name}.csv"))

All tables and `run_metadata.json` are in `MyDrive/csrp/outputs/colet/` (tables in `tables/`).